# multiply-back composite — cx3: mul_back0 vs mul_back1 — argnum dispatch (no broadcasting)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `multiply-back`, `arg-position-back-functions`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "multiply-back"
DD_ATOM_IDS = ["multiply-back", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: multiply_back", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `multiply_back0` vs `multiply_back1` — argnum dispatch

When the dispatcher hits `out = x * y` on the reverse pass, it has to choose
WHICH back-fn to call based on which parent it's propagating into:

- For parent `x` (argnum=0): chain rule gives `dL/dx = grad_out * y`.
- For parent `y` (argnum=1): chain rule gives `dL/dy = grad_out * x`.

The two bodies are NOT the same function — they use different parents. This
atom-pair isolates the **dispatch on argnum** without the unbroadcast step
(parents share the same shape here). Compare to `add`, where `back0` and
`back1` happen to have the same body — but they STILL get registered
separately at argnum=0 and argnum=1, because the dispatcher knows nothing
about which ops happen to be symmetric.


### Composite Exercise — mul_back0 vs mul_back1 — argnum dispatch (no broadcasting)

**Atoms exercised together**: `multiply-back`, `arg-position-back-functions`

Assume `x.shape == y.shape == out.shape` — no broadcasting in this drill.

Implement `cx3_mul_back(grad_out, out, x, y, argnum)` that returns:
- `grad_out * y` when `argnum == 0`,
- `grad_out * x` when `argnum == 1`,
- raises `ValueError` for any other argnum.

The argnum dispatch must select the correct *parent* to multiply against —
the test asserts the two branches are not the same function by feeding it
asymmetric `x` and `y`.


In [ ]:
def cx3_mul_back(grad_out, out, x, y, argnum):
    if argnum == 0:
        return grad_out * y            # d(x*y)/dx = y
    if argnum == 1:
        return grad_out * x            # d(x*y)/dy = x
    raise ValueError(f'argnum must be 0 or 1, got {argnum}')


<details><summary>Show solution — cx3</summary>

```python
def cx3_mul_back(grad_out, out, x, y, argnum):
    if argnum == 0:
        return grad_out * y            # d(x*y)/dx = y
    if argnum == 1:
        return grad_out * x            # d(x*y)/dy = x
    raise ValueError(f'argnum must be 0 or 1, got {argnum}')

```

The `argnum` branches encode `arg-position-back-functions`; the actual chain
rule bodies (`grad_out * y` and `grad_out * x`) encode `multiply-back`. Drop
the branching and you can't dispatch; drop the chain rule and you'd return
`grad_out` unchanged (which is `add_back`, not `multiply_back`).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx3',
        'subtopics': ["Backprop: multiply_back", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()